In [ ]:
import pandas as pd

In [2]:
# Load the cleaned datasets
transfer_df = pd.read_csv('../data/processed/transfers_clean.csv')
appearances_df = pd.read_csv('../data/processed/appearances_clean.csv')
club_games_df = pd.read_csv('../data/processed/club_games_clean.csv')
clubs_df = pd.read_csv('../data/processed/clubs_clean.csv')
players_df = pd.read_csv('../data/processed/players_clean.csv')
competitions_df = pd.read_csv('../data/processed/competitions_clean.csv')
countries_df = pd.read_csv('../data/processed/countries_clean.csv')
game_events_df = pd.read_csv('../data/processed/game_events_clean.csv')
game_lineups_df = pd.read_csv('../data/processed/game_lineups_clean.csv')
national_teams_df = pd.read_csv('../data/processed/national_teams_clean.csv')
games_df = pd.read_csv('../data/processed/games_clean.csv')
player_valuations_df = pd.read_csv('../data/processed/player_valuations_clean.csv')

C:\Users\Ali Samy\AppData\Local\Temp\ipykernel_7432\3188944719.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  game_lineups_df = pd.read_csv('../data/processed/game_lineups_clean.csv')


In [4]:
# get all unique position of players
unique_positions = players_df['position'].unique()
print("Unique player positions:", unique_positions)
unique_sub_positions = players_df['sub_position'].unique()
print("Unique player sub-positions:", unique_sub_positions)

Unique player positions: ['Attack' 'Goalkeeper' 'Defender' 'Midfield' 'Missing']
Unique player sub-positions: ['Centre-Forward' 'Goalkeeper' 'Centre-Back' 'Left-Back'
 'Attacking Midfield' 'Central Midfield' 'Defensive Midfield'
 'Left Midfield' 'Left Winger' 'Right-Back' 'Second Striker'
 'Right Winger' 'Right Midfield']


In [3]:
threshold_position_data = {
    'position' : ['Attack', 'Goalkeeper', 'Defender', 'Midfield', 'Missing'],
    'goals': [6,0,0,0,0],
    'assists': [2,2,2,6,0],
}
threshold_sub_position_data = {
    'sub_position' : ['Centre-Forward', 'Goalkeeper', 'Centre-Back', 'Left-Back','Attacking Midfield',
                      'Central Midfield', 'Defensive Midfield', 'Left Midfield', 'Left Winger',
                      'Right-Back', 'Second Striker', 'Right Winger', 'Right Midfield'],
    'goals': [0,0,0,0,2,0,0,0,1,0,3,1,0],
    'assists': [4,2,4,4,3,4,2,3,3,4,2,3,4],
}
threshold_position_df = pd.DataFrame(threshold_position_data)
threshold_sub_position_df = pd.DataFrame(threshold_sub_position_data)
player_index = players_df.set_index('player_id')


In [14]:
# This treats the combination of the two columns as a single unit
unique_pairs_count = transfer_df.set_index(['player_id', 'to_club_id']).index.nunique()

print(f"There are {unique_pairs_count} unique player-to-club transfer pairs.")

There are 121851 unique player-to-club transfer pairs.


Target Variable: Transfer Success

Definition: A transfer will be classified as 'Success' if the player meets pre-defined, measurable criteria within a 
2-3 season window after the transfer, and 'Failure' otherwise.

Success criteria may include:

Playing a minimum threshold of minutes per season

Maintaining or increasing their market value

Meeting a performance KPI specific to their position


In [11]:
# loop on all rows in transfer_df and create a new column 'transfer_success' based on the above criteria
def label_row(row):
    player_id = row['player_id']
    transfer_date = pd.to_datetime(row['transfer_date'])
    new_club_id = row['to_club_id']
    current_market_value = row['market_value_in_eur']
    
    # Get player's appearances for the new club after the transfer date
    # 1. Calculate the end date (Transfer Date + 3 Years)
    three_seasons_end = pd.to_datetime(transfer_date) + pd.DateOffset(years=3)

    # 2. Add the end date condition to your filter
    appearances = appearances_df[
        (appearances_df['player_id'] == player_id) &
        (appearances_df['player_club_id'] == new_club_id) &
        (pd.to_datetime(appearances_df['date']) >= transfer_date) &
        (pd.to_datetime(appearances_df['date']) < three_seasons_end)
    ]
    total_minutes = appearances["minutes_played"].sum()
    total_goals = appearances["goals"].sum()
    total_assists = appearances["assists"].sum()
    # get player's position
    #player_position = players_df.loc[players_df["player_id"] == player_id,"position"].values[0]
    player_position = player_index.at[player_id, "position"]
    player_sub_position = player_index.at[player_id, "sub_position"]
    
    # Get player's valuations for the new club after the transfer date
    valuations = player_valuations_df[
        (player_valuations_df['player_id'] == player_id) &
        (player_valuations_df['current_club_id'] == new_club_id) &
        (pd.to_datetime(player_valuations_df['date']) >= transfer_date) &
        (pd.to_datetime(player_valuations_df['date']) < three_seasons_end)
    ]
    
    new_market_value = valuations["market_value_in_eur"].max()  # Get the maximum market value after the transfer

    scores = [] # List to store scores for each criterion
    # 1. Playing Time Criterion
    if total_minutes >= 1800:  # 90 minutes * 20 games
        scores.append(1)
    else:
        scores.append(0)
    # 2. Performance Criterion
    player_threshold_position = threshold_position_df[threshold_position_df['position'] == player_position]
    player_threshold_sub_position = threshold_sub_position_df[threshold_sub_position_df['sub_position'] == player_sub_position]
    if total_assists >= player_threshold_position['assists'].values[0]:
        scores.append(1)
    else:
        scores.append(0)
    if total_goals >= player_threshold_position['goals'].values[0]:
        scores.append(1)
    else:         
        scores.append(0)
    if total_assists >= player_threshold_sub_position['assists'].values[0]:
        scores.append(1)
    else:
        scores.append(0)
    if total_goals >= player_threshold_sub_position['goals'].values[0]:
        scores.append(1)
    else:         
        scores.append(0)
    # 3. Market Value Criterion
    if new_market_value > current_market_value:
        scores.append(1)
    else:
        scores.append(0)

    # Calculate the overall score each criterion is equally weighted
    overall_score = sum(scores) / len(scores)

    # Label the transfer based on the overall score
    if overall_score >= 0.5:
        return 1  # Successful transfer
    else:
        return 0  # Unsuccessful transfer

In [4]:
# PRE-STEP: Ensure all date columns are datetime objects (CRITICAL)
appearances_df['date'] = pd.to_datetime(appearances_df['date'])
player_valuations_df['date'] = pd.to_datetime(player_valuations_df['date'])
transfer_df['transfer_date'] = pd.to_datetime(transfer_df['transfer_date'])

# 1. PRE-CALCULATE AGGREGATES
# Instead of filtering inside a loop, we merge appearances and transfers
# to calculate stats for all transfers at once.
temp_app = transfer_df[['player_id', 'to_club_id', 'transfer_date']].merge(
    appearances_df, 
    left_on=['player_id', 'to_club_id'], 
    right_on=['player_id', 'player_club_id']
)

# Apply the 3-season logic via boolean masking
temp_app['end_date'] = temp_app['transfer_date'] + pd.DateOffset(years=3)
mask = (temp_app['date'] >= temp_app['transfer_date']) & (temp_app['date'] < temp_app['end_date'])
app_stats = temp_app[mask].groupby(['player_id', 'transfer_date']).agg({
    'minutes_played': 'sum',
    'goals': 'sum',
    'assists': 'sum'
}).reset_index()

# 2. PRE-CALCULATE MARKET VALUE MAX
temp_val = transfer_df[['player_id', 'to_club_id', 'transfer_date']].merge(
    player_valuations_df,
    left_on=['player_id', 'to_club_id'],
    right_on=['player_id', 'current_club_id']
)
val_mask = (temp_val['date'] >= temp_val['transfer_date']) & (temp_val['date'] < (temp_val['transfer_date'] + pd.DateOffset(years=3)))
max_vals = temp_val[val_mask].groupby(['player_id', 'transfer_date'])['market_value_in_eur'].max().reset_index(name='new_market_value')

# 3. ASSEMBLE THE MAIN DATAFRAME
final_df = transfer_df.merge(app_stats, on=['player_id', 'transfer_date'], how='left')
final_df = final_df.merge(max_vals, on=['player_id', 'transfer_date'], how='left')
final_df = final_df.merge(players_df[['player_id', 'position', 'sub_position']], on='player_id', how='left')

# 4. ATTACH THRESHOLDS
final_df = final_df.merge(threshold_position_df, on='position', how ='left',suffixes=('', '_pos_limit'))
final_df = final_df.merge(threshold_sub_position_df, on='sub_position', how ='left', suffixes=('', '_sub_limit'))

# Fill NaNs with 0 (for players with no appearances/valuations)
final_df[['minutes_played', 'goals', 'assists', 'new_market_value']] = final_df[['minutes_played', 'goals', 'assists', 'new_market_value']].fillna(0)

# 5. THE VECTORIZED COMPARISON (The "Scores")
# No "if" statements needed! Booleans (True/False) convert to 1/0 automatically.
s1 = (final_df['minutes_played'] >= 1800).astype(int)
s2 = (final_df['assists'] >= final_df['assists_pos_limit']).astype(int)
s3 = (final_df['goals'] >= final_df['goals_pos_limit']).astype(int)
s4 = (final_df['assists'] >= final_df['assists_sub_limit']).astype(int)
s5 = (final_df['goals'] >= final_df['goals_sub_limit']).astype(int)
s6 = (final_df['new_market_value'] > final_df['market_value_in_eur']).astype(int)

# 6. FINAL SUCCESS CALCULATION
final_df['overall_score'] = (s1*0.3 + s2*0.1 + s3*0.1 + s4*0.1 + s5*0.1 + s6*0.3)
final_df['transfer_success'] = (final_df['overall_score'] >= 0.5).astype(int)
final_df.to_csv("../data/processed/labeled_transfers.csv", index=False)

In [12]:
transfer_labeled_df = transfer_df.copy()
transfer_labeled_df['transfer_success'] = transfer_labeled_df.apply(label_row, axis=1)
transfer_labeled_df.to_csv('../data/processed/transfers_labeled.csv')

KeyboardInterrupt: 

In [5]:
print("Target Distribution:")
print(final_df['transfer_success'].value_counts())

print("\nPercentage:")
print(final_df['transfer_success'].value_counts(normalize=True) * 100)

Target Distribution:
transfer_success
0    116813
1     40373
Name: count, dtype: int64

Percentage:
transfer_success
0    74.315143
1    25.684857
Name: proportion, dtype: float64
